# Football Shots Model — exploration

A workbench for poking at the database: sanity-checking a player's history, looking at head-to-head records, checking scrape coverage. Run it locally (Jupyter / VS Code) from the repo root.

The data itself is loaded by `python scripts/backfill.py` (one-time, resumable, on your own PC) and kept current by the weekly GitHub Actions job — not from here.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

import pandas as pd
import config
from src import db

players = db.load_player_match_stats()
teams = db.load_team_match_stats()
print(f'{len(players):,} player-match rows, {players.match_id.nunique():,} matches')

## Scrape coverage by season and competition

In [ ]:
db.scrape_progress().pivot_table(index=['season', 'competition'], columns='details_status',
                                  values='n', fill_value=0, aggfunc='sum')

## One player's match-by-match shot log

In [ ]:
name = 'Haaland'
cols = ['date', 'competition', 'team', 'opponent', 'started', 'minutes', 'shots', 'shots_on_target', 'xg']
players[players.player_name.str.contains(name, case=False)].sort_values('date')[cols].tail(20)

## Head-to-head between two teams, across seasons

In [ ]:
a, b = 'Real Madrid', 'Atletico Madrid'
h2h = teams[(teams.team.str.contains(a, case=False)) & (teams.opponent.str.contains(b, case=False))]
h2h.sort_values('date')[['date', 'season', 'competition', 'is_home', 'goals_for', 'goals_against',
                          'shots_for', 'shots_against', 'xg_for', 'possession', 'coach_name']]